# ⛓️ Chains & Runnables (LCEL) — Hands-On Examples

> **Module 03 | Submodule 04 | Practical Notebook**
>
> Sections:
> 1. Your first LCEL chain
> 2. All Runnable methods: invoke, stream, batch, async
> 3. RunnableParallel — parallel chains
> 4. RunnablePassthrough & RunnableLambda
> 5. Sequential chain pattern
> 6. Parallel chain + synthesis
> 7. Conditional routing chain
> 8. Iterative refinement chain
> 9. Fallback chain

In [ ]:
!pip install langchain langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "module-03-chains-runnables"

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Ready!")

---
## 1️⃣ Your First LCEL Chain

In [ ]:
# The simplest LCEL chain: prompt | llm | parser
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep answers to 2 sentences."),
    ("human", "What is {concept}?")
])
parser = StrOutputParser()

chain = prompt | llm | parser

# What type is chain?
print("Chain type:", type(chain).__name__)   # RunnableSequence

# Use it
result = chain.invoke({"concept": "LangChain"})
print("Result type:", type(result).__name__)  # str
print("Result:", result)

In [ ]:
# Inspect the chain
print("Input schema:",  chain.input_schema.schema())
print("Output schema:", chain.output_schema.schema())

# Visualize
print("\nChain graph:")
chain.get_graph().print_ascii()

---
## 2️⃣ All Runnable Methods

In [ ]:
# .invoke() — single call
result = chain.invoke({"concept": "LCEL"})
print("invoke() →", result[:60], "...")

In [ ]:
# .stream() — token by token
print("stream() tokens:")
for chunk in chain.stream({"concept": "LangGraph"}):
    print(chunk, end="", flush=True)
print()

In [ ]:
import time

# .batch() — parallel processing
inputs = [
    {"concept": "LangChain"},
    {"concept": "LangGraph"},
    {"concept": "LangSmith"},
    {"concept": "RAG"},
]

# Sequential (slow)
start = time.time()
seq_results = [chain.invoke(inp) for inp in inputs]
seq_time = time.time() - start

# Batch/parallel (fast)
start = time.time()
batch_results = chain.batch(inputs)
batch_time = time.time() - start

print(f"Sequential: {seq_time:.2f}s")
print(f"Batch:      {batch_time:.2f}s")
print(f"Speedup:    {seq_time/batch_time:.1f}x faster")

In [ ]:
# .ainvoke() — async
import asyncio

async def run_async():
    result = await chain.ainvoke({"concept": "async LangChain"})
    print("ainvoke():", result[:60])

    # astream — async streaming
    print("astream():")
    async for chunk in chain.astream({"concept": "async streaming"}):
        print(chunk, end="", flush=True)
    print()

await run_async()  # Works directly in Jupyter notebooks

---
## 3️⃣ RunnableParallel

In [ ]:
from langchain_core.runnables import RunnableParallel

# Two chains to run in parallel
pros_chain = (
    ChatPromptTemplate.from_template("List 3 pros of {topic} (bullet points)")
    | llm | StrOutputParser()
)
cons_chain = (
    ChatPromptTemplate.from_template("List 3 cons of {topic} (bullet points)")
    | llm | StrOutputParser()
)
example_chain = (
    ChatPromptTemplate.from_template("Give 2 real use cases for {topic}")
    | llm | StrOutputParser()
)

parallel = RunnableParallel({
    "pros":     pros_chain,
    "cons":     cons_chain,
    "examples": example_chain,
})

import time
start = time.time()
result = parallel.invoke({"topic": "LangChain"})
print(f"Elapsed: {time.time() - start:.2f}s (3 LLM calls in parallel)\n")

for key, val in result.items():
    print(f"{'='*30}\n{key.upper()}:\n{val}\n")

---
## 4️⃣ RunnablePassthrough & RunnableLambda

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# RunnablePassthrough — passes input unchanged
passthrough = RunnablePassthrough()
result = passthrough.invoke({"question": "What is LangChain?", "context": "Some docs..."})
print("Passthrough:", result)  # Same dict as input

In [ ]:
# RunnablePassthrough.assign() — add fields to dict
enricher = RunnablePassthrough.assign(
    question_upper=lambda x: x["question"].upper(),
    word_count    =lambda x: len(x["question"].split()),
)

result = enricher.invoke({"question": "What is LangChain?"})
print(result)
# {'question': 'What is LangChain?',
#  'question_upper': 'WHAT IS LANGCHAIN?',
#  'word_count': 4}

In [ ]:
# RunnableLambda — any Python function as a Runnable
reverse = RunnableLambda(lambda x: x[::-1])
print("Reversed:", reverse.invoke("LangChain"))  # "niahCgnaL"

# Use in a chain
word_count_chain = (
    ChatPromptTemplate.from_template("Write about {topic} in exactly one paragraph.")
    | llm
    | StrOutputParser()
    | RunnableLambda(lambda text: {
        "text": text,
        "words": len(text.split()),
        "chars": len(text)
    })
)

result = word_count_chain.invoke({"topic": "LangChain"})
print(f"Words: {result['words']}, Chars: {result['chars']}")
print(f"Text: {result['text'][:80]}...")

---
## 5️⃣ Sequential Chain — Multi-Step Pipeline

In [ ]:
# Multi-step: Topic → Outline → Blog Post
outline_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Create a focused 3-point outline. Output only the outline."),
        ("human",  "Create outline for: {topic}")
    ]) | llm | StrOutputParser()
)

write_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a skilled blog writer. Write engagingly."),
        ("human",  "Write a 3-paragraph blog post from this outline:\n{outline}")
    ]) | llm | StrOutputParser()
)

# Chain them sequentially
full_chain = (
    outline_chain
    | (lambda outline: {"outline": outline})  # Reshape output for next step
    | write_chain
)

print("Generating blog post...")
blog = full_chain.invoke({"topic": "Why LangChain is changing AI development"})
print(blog)

---
## 6️⃣ Parallel + Synthesis

In [ ]:
from langchain_core.runnables import RunnableParallel

# Gather multiple perspectives in parallel
gather = RunnableParallel({
    "pros":   ChatPromptTemplate.from_template("3 pros of {topic}") | llm | StrOutputParser(),
    "cons":   ChatPromptTemplate.from_template("3 cons of {topic}") | llm | StrOutputParser(),
    "future": ChatPromptTemplate.from_template("Future of {topic} in 2 sentences") | llm | StrOutputParser(),
    "topic":  lambda x: x["topic"]   # Pass topic through
})

# Synthesize into one analysis
synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a balanced analysis given these inputs."),
    ("human", """Topic: {topic}

Pros: {pros}

Cons: {cons}

Future: {future}

Synthesize into a 3-sentence balanced analysis:""")
])

# Full chain: parallel gather → synthesis
analysis_chain = gather | synthesis_prompt | llm | StrOutputParser()

result = analysis_chain.invoke({"topic": "LangChain"})
print(result)

---
## 7️⃣ Conditional Routing Chain

In [ ]:
from langchain_core.runnables import RunnableLambda

# Specialized chains
code_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a Python expert. Give concise, working code examples."),
        ("human",  "{question}")
    ]) | llm | StrOutputParser()
)

explain_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Explain concepts clearly for beginners. Use analogies."),
        ("human",  "{question}")
    ]) | llm | StrOutputParser()
)

compare_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Create a clear comparison table."),
        ("human",  "{question}")
    ]) | llm | StrOutputParser()
)

# Router
def route(inputs):
    question = inputs["question"].lower()
    if any(w in question for w in ["code", "implement", "write", "example in python"]):
        print("📝 Routing to: code_chain")
        return code_chain.invoke(inputs)
    elif any(w in question for w in ["vs", "difference", "compare", "better"]):
        print("⚖️  Routing to: compare_chain")
        return compare_chain.invoke(inputs)
    else:
        print("💡 Routing to: explain_chain")
        return explain_chain.invoke(inputs)

router = RunnableLambda(route)

# Test routing
test_questions = [
    "Write a simple LangChain chain in Python",
    "What is the difference between LangChain and LangGraph?",
    "What is LCEL and why should I use it?",
]

for q in test_questions:
    print(f"\nQ: {q}")
    ans = router.invoke({"question": q})
    print(f"A: {ans[:150]}...")

---
## 8️⃣ Iterative Refinement Chain

In [ ]:
from pydantic import BaseModel, Field

class Evaluation(BaseModel):
    score: int = Field(description="Quality score 1-10")
    is_good_enough: bool = Field(description="True if score is 7 or above")
    improvement: str = Field(description="One specific improvement to make")

# Generator
writer = (
    ChatPromptTemplate.from_messages([
        ("system", "Write a catchy one-liner tagline. If given feedback, incorporate it."),
        ("human",  "Product: {product}\nFeedback: {feedback}")
    ]) | llm | StrOutputParser()
)

# Evaluator
evaluator = (
    ChatPromptTemplate.from_messages([
        ("system", "Evaluate this marketing tagline. Score 1-10 based on: clarity, impact, memorability."),
        ("human",  "Product: {product}\nTagline: {tagline}")
    ]) | llm.with_structured_output(Evaluation)
)

def refine_until_good(product: str, max_iter: int = 4):
    feedback = "No feedback yet. Write your best tagline."
    tagline  = ""

    for i in range(max_iter):
        tagline = writer.invoke({"product": product, "feedback": feedback})
        eval_   = evaluator.invoke({"product": product, "tagline": tagline})

        print(f"Iteration {i+1}: \"{tagline}\" → Score: {eval_.score}/10")
        if eval_.is_good_enough:
            print("✅ Good enough!")
            break
        feedback = eval_.improvement

    return tagline

final_tagline = refine_until_good("LangChain — a Python framework for AI apps")
print(f"\n🏆 Final tagline: '{final_tagline}'")

---
## 9️⃣ Fallback Chain

In [ ]:
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template("In one sentence, what is {topic}?")

# Primary: gpt-4o (expensive)
primary = prompt | ChatOpenAI(model="gpt-4o", temperature=0) | StrOutputParser()

# Fallback: gpt-4o-mini (cheaper) — triggers if primary fails
fallback = prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | StrOutputParser()

# Chain with automatic fallback
resilient = primary.with_fallbacks([fallback])
result = resilient.invoke({"topic": "LangChain"})
print(result)

In [ ]:
# .with_retry() — automatic retry on failure
from langchain_openai import ChatOpenAI

reliable_chain = (
    prompt
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
).with_retry(
    stop_after_attempt=3,          # Try up to 3 times
    wait_exponential_jitter=True,  # Wait between retries
)

result = reliable_chain.invoke({"topic": "LCEL"})
print(result)

---
## ✅ Summary — LCEL Cheat Sheet

```python
# Basic chain
chain = prompt | llm | parser

# All methods
chain.invoke(input)           # Single call
chain.stream(input)           # Stream tokens
chain.batch([i1, i2, i3])    # Parallel batch
await chain.ainvoke(input)   # Async

# Parallel
parallel = {"a": chainA, "b": chainB} | next_step

# Pass-through
{"context": retriever, "q": RunnablePassthrough()}

# Lambda transform
chain | RunnableLambda(lambda x: {"key": x})

# Resilience
chain.with_retry(stop_after_attempt=3)
chain.with_fallbacks([backup_chain])
```

**Next**: [05 — Document Loaders & Text Splitters](../05_document_loaders_and_text_splitters/theory.md)